
# 🎧 Audio Regression on Mini Speech Commands (Colab-Ready)
**What this notebook does**  
- Downloads **Mini Speech Commands** dataset (open-source subset of Google Speech Commands)  
- Extracts **audio features**: MFCCs, chroma, zero-crossing rate (ZCR), RMS energy  
- Creates a **synthetic regression target** (e.g., loudness proxy)  
- Trains/evaluates multiple regressors: **Linear**, **Ridge**, **Random Forest**, **k-NN**  
- Produces **figures** (predicted vs actual, residual plots, error distribution) and a **comparison table**  
- Written to be **educational** with rich comments and easy to tweak for class demos

> Tip: In Google Colab, you can run this end-to-end. If you want to save outputs to Drive, add the Drive mount cell.


In [ ]:

# # (Optional) Mount Google Drive to save results (uncomment in Colab)
# from google.colab import drive
# drive.mount('/content/drive')

# Install audio libs if needed (Colab usually has these, but leaving here for portability)
# If you see import errors later, uncomment and run:
# !pip install librosa soundfile


In [ ]:

import os, zipfile, urllib.request, glob, io, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import librosa

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score



## 1) Download the Mini Speech Commands dataset
A small, open subset of Google Speech Commands: ~1-second WAV clips of spoken words (e.g., *yes, no, up, down, go*).


In [ ]:

DATA_URL = "http://storage.googleapis.com/download.tensorflow.org/data/mini_speech_commands.zip"
DATA_ZIP = "mini_speech_commands.zip"
DATA_DIR = "mini_speech_commands"

if not os.path.exists(DATA_DIR):
    print("Downloading dataset...")
    urllib.request.urlretrieve(DATA_URL, DATA_ZIP)
    print("Extracting...")
    with zipfile.ZipFile(DATA_ZIP, 'r') as zf:
        zf.extractall(".")
    print("Done.")
else:
    print("Dataset already present.")

labels = [d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d)) and not d.startswith("_")]
labels.sort()
labels



## 2) Parameters & file listing
Limit files per label to keep runtime fast for class demonstration. Increase for more data.


In [ ]:

SAMPLE_RATE = 16000            # dataset sample rate
MAX_FILES_PER_LABEL = 100      # change as needed (trade-off: runtime vs. accuracy)

files, y_labels = [], []
for lab in labels:
    paths = glob.glob(os.path.join(DATA_DIR, lab, "*.wav"))
    paths = sorted(paths)[:MAX_FILES_PER_LABEL]
    for p in paths:
        files.append(p)
        y_labels.append(lab)

print(f"Collected {len(files)} files across {len(labels)} labels.")
print("Example files:", files[:3])



## 3) Feature extraction
We compute a compact feature vector per clip:
- **MFCCs** (13 coefficients) → mean & std across time (26 features)
- **Chroma** (12-D) → mean & std (24 features)
- **Zero-Crossing Rate** → mean & std (2 features)
- **RMS energy** → mean & std (2 features)

**Total features:** 26 + 24 + 2 + 2 = **54**


In [ ]:

def extract_features(y, sr):
    """Return a 54-D feature vector for an audio clip.
    Features:
      - MFCCs (13): mean & std -> 26
      - Chroma (12): mean & std -> 24
      - ZCR: mean & std -> 2
      - RMS: mean & std -> 2
    """
    feats = []

    # Ensure 1-second clips by padding/truncating (dataset is ~1s already, but just in case)
    target_len = sr  # 1 second * sr
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)))
    else:
        y = y[:target_len]

    # 1) MFCCs
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    mfcc_mean = mfcc.mean(axis=1)
    mfcc_std  = mfcc.std(axis=1)
    feats.extend(np.concatenate([mfcc_mean, mfcc_std], axis=0))

    # 2) Chroma
    chroma = librosa.feature.chroma_stft(y=y, sr=sr, n_chroma=12)
    chroma_mean = chroma.mean(axis=1)
    chroma_std  = chroma.std(axis=1)
    feats.extend(np.concatenate([chroma_mean, chroma_std], axis=0))

    # 3) Zero-Crossing Rate
    zcr = librosa.feature.zero_crossing_rate(y=y)
    feats.append(zcr.mean())
    feats.append(zcr.std())

    # 4) RMS energy
    rms = librosa.feature.rms(y=y)
    feats.append(rms.mean())
    feats.append(rms.std())

    return np.asarray(feats, dtype=np.float32)



### Build feature matrix `X`
Loop over WAVs, load with `librosa`, extract features, and stack into a NumPy array.


In [ ]:

X = []
for fp in files:
    y, sr = librosa.load(fp, sr=SAMPLE_RATE)
    X.append(extract_features(y, sr))
X = np.vstack(X)
y_labels = np.array(y_labels)
X.shape



## 4) Create a synthetic regression target
For demonstration, predict a continuous **loudness proxy** from features.  
We use the **mean RMS energy** (already included in our features) with a mild non-linear transform to make the task interesting:
\[ y = 10 \cdot \sqrt{\text{RMS\_mean}} \]


In [ ]:

# Index of RMS mean in feature vector: after MFCC(26) + Chroma(24) + ZCR(2) = 52 -> RMS mean at 52, RMS std at 53
RMS_MEAN_IDX = 52
raw_rms_mean = X[:, RMS_MEAN_IDX]
y_reg = np.sqrt(np.clip(raw_rms_mean, 0, None)) * 10.0

print("Target stats -> min:", float(y_reg.min()), "max:", float(y_reg.max()), "mean:", float(y_reg.mean()))



## 5) Train/Test split
We keep 20% for testing to evaluate generalization.


In [ ]:

X_train, X_test, y_train, y_test = train_test_split(X, y_reg, test_size=0.2, random_state=42)
X_train.shape, X_test.shape



## 6) Train multiple regressors
- **Linear Regression** (OLS)
- **Ridge Regression** (L2 regularization)
- **Random Forest Regressor** (nonlinear ensemble)
- **k-NN Regressor** (instance-based)


In [ ]:

lin_reg  = LinearRegression()
ridge    = Ridge(alpha=1.0)
rf       = RandomForestRegressor(n_estimators=100, random_state=42)
knn      = KNeighborsRegressor(n_neighbors=5)

models = {
    "Linear": lin_reg,
    "Ridge": ridge,
    "RandomForest": rf,
    "KNN": knn
}

for name, mdl in models.items():
    mdl.fit(X_train, y_train)
print("Models trained.")



## 7) Evaluation: MAE, MSE, RMSE, $R^2$
Create a comparison table and display it nicely.


In [ ]:

def evaluate(mdl, X_te, y_te):
    pred = mdl.predict(X_te)
    mae = mean_absolute_error(y_te, pred)
    mse = mean_squared_error(y_te, pred)
    rmse = math.sqrt(mse)
    r2  = r2_score(y_te, pred)
    return mae, mse, rmse, r2, pred

rows = []
pred_store = {}
for name, mdl in models.items():
    mae, mse, rmse, r2, pred = evaluate(mdl, X_test, y_test)
    rows.append([name, mae, mse, rmse, r2])
    pred_store[name] = pred

results_df = pd.DataFrame(rows, columns=["Model", "MAE", "MSE", "RMSE", "R2"])
results_df.sort_values("R2", ascending=False, inplace=True)
results_df.reset_index(drop=True, inplace=True)
results_df



## 8) Visualizations
### a) Predicted vs Actual (best model)


In [ ]:

# Choose best model by R2
best_model_name = results_df.iloc[0]["Model"]
best_pred = pred_store[best_model_name]

plt.figure(figsize=(6,5))
plt.scatter(y_test, best_pred, alpha=0.7)
lims = [min(y_test.min(), best_pred.min()), max(y_test.max(), best_pred.max())]
plt.plot(lims, lims, 'r--', label='Ideal')
plt.xlabel("Actual Target")
plt.ylabel("Predicted Target")
plt.title(f"Predicted vs Actual — {best_model_name}")
plt.legend()
plt.tight_layout()
plt.show()



### b) Residual analysis (best model)
Residuals should be centered around 0 without strong patterns.


In [ ]:

resid = y_test - best_pred

plt.figure(figsize=(6,4))
plt.scatter(best_pred, resid, alpha=0.7, color='green')
plt.axhline(0, color='red', linestyle='--')
plt.xlabel("Predicted Target")
plt.ylabel("Residual (Actual - Predicted)")
plt.title(f"Residuals vs Predicted — {best_model_name}")
plt.tight_layout()
plt.show()

plt.figure(figsize=(6,4))
plt.hist(resid, bins=15, color='purple', edgecolor='black')
plt.axvline(resid.mean(), color='red', linestyle='--', label=f"Mean={resid.mean():.3f}")
plt.xlabel("Residual")
plt.ylabel("Frequency")
plt.title(f"Residual Distribution — {best_model_name}")
plt.legend()
plt.tight_layout()
plt.show()



### c) Results table (CSV + styled display)


In [ ]:

# Save comparison table
results_path = "audio_regression_results.csv"
results_df.to_csv(results_path, index=False)
print("Saved:", results_path)

# Display styled table with highlighting of best R2 (just for notebook view)
def highlight_best(s):
    is_best = s == s.max()
    return ['background-color: #d0f0c0' if v else '' for v in is_best]

results_df.style.format({
    "MAE": "{:.3f}", "MSE": "{:.3f}", "RMSE": "{:.3f}", "R2": "{:.3f}"
}).apply(highlight_best, subset=["R2"])



## 9) Notes / Extensions
- Try increasing `MAX_FILES_PER_LABEL` for better-trained models (longer runtime).
- Standardize/scale features and re-run k-NN (distance-sensitive) to compare.
- Add **PolynomialFeatures** to Linear/Ridge for non-linear trends.
- Try **SVR** or **Gradient Boosting Regressor** for more baselines.
- Replace synthetic target with a real continuous property (e.g., dB SPL if available).
